# Natural Language Understanding

In this section we go through topics related to text understanding. We cover such topics like:
    
- Similarity measures
- Word Vectors
- Vector Space Model
- Type of vectorizers
- Build a vectorizer with Tensorflow.

## Similarity measures

Word does have different meanings. This makes the comparison and analysis a bit more complex.

In [2]:
from textblob import Word

w = Word("developer")

for synset, definition in zip(w.get_synsets(), w.define()):
    print(synset, definition)

Synset('developer.n.01') someone who develops real estate (especially someone who prepares a site for residential or commercial use)
Synset('developer.n.02') photographic equipment consisting of a chemical solution for developing film


## Similarity measures

There are plenty of methods to measure the similarity of strings. Two most popular Python libraries examples for such measure are shown. We compare two strings: trains and training. The SequenceMatcher class allow us to use the Gestalt pattern matching algorithm:

In [3]:
from difflib import SequenceMatcher
a = "training"
b = "trains"
print(len(a))
print(len(b))
ratio = SequenceMatcher(None, a, b).ratio()
print(ratio)

8
6
0.7142857142857143


The distance is a normalized value between 0 and 1, where 1 means identical.

A different approach is shown below. We use the Jellyfish library. There are a few methods that we can use here. One of it is the Levenshtein distance. Below the distance and normalize distance values are calculated.

In [5]:
import jellyfish
distance = jellyfish.levenshtein_distance(a,b)
print(distance)

normalized_distance = distance/max(len(a),len(b))
print(1.0-normalized_distance)

3
0.625


Some words can be more similar to each other than other. We can build a similarity matrix to check it where 1 mean equal and 0 totally different.

In [6]:
import spacy

nlp = spacy.load("en_core_web_sm")

tokens = nlp(u'king queen horse cat desk lamp')

for first_token in tokens:
    for second_token in tokens:
        print(first_token.text, second_token.text, first_token.similarity(second_token))

king king 1.0
king queen 0.5523890256881714
king horse 0.6490846872329712
king cat 0.6355116963386536
king desk 0.621141254901886
king lamp 0.3553647994995117
queen king 0.5523890256881714
queen queen 1.0
queen horse 0.6413618326187134
queen cat 0.6422978043556213
queen desk 0.4828203618526459
queen lamp 0.28534623980522156
horse king 0.6490846872329712
horse queen 0.6413618326187134
horse horse 1.0
horse cat 0.7530450224876404
horse desk 0.7113964557647705
horse lamp 0.2819593846797943
cat king 0.6355116963386536
cat queen 0.6422978043556213
cat horse 0.7530450224876404
cat cat 1.0
cat desk 0.7171131372451782
cat lamp 0.2658533453941345
desk king 0.621141254901886
desk queen 0.4828203618526459
desk horse 0.7113964557647705
desk cat 0.7171131372451782
desk desk 1.0
desk lamp 0.351275235414505
lamp king 0.3553647994995117
lamp queen 0.28534623980522156
lamp horse 0.2819593846797943
lamp cat 0.2658533453941345
lamp desk 0.351275235414505
lamp lamp 1.0


/var/folders/ly/t71lcvj913l2v610608ynmdr0000gn/T/ipykernel_37986/4273872675.py:9: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Token.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your own word vectors, or use one of the larger models instead if available.
  print(first_token.text, second_token.text, first_token.similarity(second_token))


We can also compare sentences:

In [15]:
doc1 = nlp(u"Warsaw is the largest city in Poland.")
doc2 = nlp(u"Crossaint is baked in France.")
doc3 = nlp(u"An emu is a large bird.")

for doc in [doc1, doc2, doc3]:
    for other_doc in [doc1, doc2, doc3]:
        print(doc[:1], other_doc[:1], doc.similarity(other_doc))

Warsaw Warsaw 1.0
Warsaw Crossaint 0.7503141760826111
Warsaw An 0.6300676465034485
Crossaint Warsaw 0.7503141760826111
Crossaint Crossaint 1.0
Crossaint An 0.5422711372375488
An Warsaw 0.6300676465034485
An Crossaint 0.5422711372375488
An An 1.0


/var/folders/ly/t71lcvj913l2v610608ynmdr0000gn/T/ipykernel_37986/2224194687.py:7: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Doc.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your own word vectors, or use one of the larger models instead if available.
  print(doc[:1], other_doc[:1], doc.similarity(other_doc))


The similarity matrix looks like following:

|       | doc1 | doc2 | doc3 |
|-------|------|------|------|
| **doc1** | 1.0  | 0.72 | 0.65 |
| **doc 2** | 0.72 | 1.0  | 0.40 |
| **doc 3** | 0.65 | 0.40 | 1.0  |

## Word Vectors

SpaCy does have already a set of words that are vectorized.

Let's take a look at the vectors that are available in spaCy using the previous example:

In [17]:
nlp = spacy.load("en_core_web_sm")

tokens = nlp(u'king queen horse cat desk lamp')

for token in tokens:
    print(str(token)+" "+str(token.vector))

king [-0.33849967 -0.03223152  0.29518685  0.5957103   0.22842334 -0.24313475
  1.2632762   0.4088267   0.29961005  0.38803428  0.48059282 -0.52212435
 -0.566893    0.21759084 -0.4746175   0.15028849 -0.06043254  0.17063619
  1.1439679  -1.3166866  -0.2312963   1.1934105  -0.84843856  0.5214447
  1.3578575   0.23143815 -0.17714164  0.5807784  -0.78381205  0.28310817
 -0.5557947  -0.5037913  -0.4577139  -0.22190148 -1.1596107  -0.31535238
 -0.6977457   0.18609415 -0.46583927  1.1962727  -0.15789706  1.009355
 -0.95039606  0.999201   -0.09383057  0.61801815 -0.8498037   0.22529945
 -0.9754846  -0.57056195 -0.4995727   0.2543295  -0.18943411 -0.65711564
 -0.21509793 -0.8799943   0.48541635  0.61795837  0.4118355   0.32871026
 -0.8590191  -0.1063208   0.34067032 -0.9123617  -1.270268   -0.16583705
 -0.11312231  1.4501243  -0.08616476 -1.0183003   0.36684948  0.71151984
  0.81917906 -0.24078491  0.91655225 -0.6053557  -0.68099177 -0.679268
  0.9947961   0.5872688  -0.3395236  -0.4532945  -0

It looks that the vectors are quite long. It's easy to check the exact size of a vector:

In [18]:
len(tokens[1].vector)

96

You can play around and check the vector values for some other sentences. Let's take a look at sentence vectors of one of our previous examples:

In [19]:
len(doc1.vector)

96

A nice example of word vectorization done by some researchers at Warsaw University: [Word2Vec](https://lamyiowce.github.io/word2viz/).

## Negative sampling

It is a simpler implementation of word2vec. It is faster as it takes only a few terms in each iteration for training insted of the whole dataset as in previous example. This is why it's called negative sampling.

First of all, we define helper methods that are used later.

In [20]:
def zeros(*dims):
    return np.zeros(shape=tuple(dims), dtype=np.float32)

def ones(*dims):
    return np.ones(shape=tuple(dims), dtype=np.float32)

def rand(*dims):
    return np.random.rand(*dims).astype(np.float32)

def randn(*dims):
    return np.random.randn(*dims).astype(np.float32)

def sigmoid(batch, stochastic=False):
    return  1.0 / (1.0 + np.exp(-batch))

def as_matrix(vector):
    return np.reshape(vector, (-1, 1))

We need to load the data again.

In [21]:
import nltk
import numpy as np
import pandas as pd
from collections import namedtuple

nltk.download('all')

from nltk.book import *

texts()

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /Users/igorz/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to
[nltk_data]    |     /Users/igorz/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /Users/igorz/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /Users/igorz/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /Users/igorz/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_ru is already
[nltk_data]    |       up-to-date!
[nltk_dat

*** Introductory Examples for the NLTK Book ***
Loading text1, ..., text9 and sent1, ..., sent9
Type the name of the text or sentence to view it.
Type: 'texts()' or 'sents()' to list the materials.
text1: Moby Dick by Herman Melville 1851
text2: Sense and Sensibility by Jane Austen 1811
text3: The Book of Genesis
text4: Inaugural Address Corpus
text5: Chat Corpus
text6: Monty Python and the Holy Grail
text7: Wall Street Journal
text8: Personals Corpus
text9: The Man Who Was Thursday by G . K . Chesterton 1908
text1: Moby Dick by Herman Melville 1851
text2: Sense and Sensibility by Jane Austen 1811
text3: The Book of Genesis
text4: Inaugural Address Corpus
text5: Chat Corpus
text6: Monty Python and the Holy Grail
text7: Wall Street Journal
text8: Personals Corpus
text9: The Man Who Was Thursday by G . K . Chesterton 1908


Three variables are important for the training: ``train_dict``, ``train_tokens`` and ``train_set``. The first one contain all unique words used in the corpus. The second is a list of indices of words in the dictionary that correspond to each word used in the raw text. 

In [22]:
#raw_set = nltk.corpus.treebank_raw.raw()[0:50000].replace('.START',' ').replace("\n","").replace("."," ").replace(","," ")
#tokens = [token for token in nltk.word_tokenize(raw_set) if token.isalpha()]
tokens = text6.tokens
train_dict = pd.Series(tokens).unique().tolist()
train_tokens = np.array([train_dict.index(token) for token in tokens])

The last variable consist of a list of two numbers. The current word index and the word index that is before the word and after the word. Depending on the window size we use also other words that are in the neighbourhood. In this example the window size is set to 2. It means we take two words before and two words after the given word and build the relation in the training data set.

In [23]:
train_set = []
for i in range(2,len(tokens)-2):
    train_set.append([train_dict.index(tokens[i]), train_dict.index(tokens[i-1])])
    train_set.append([train_dict.index(tokens[i]), train_dict.index(tokens[i-2])])
    train_set.append([train_dict.index(tokens[i]), train_dict.index(tokens[i+1])])
    train_set.append([train_dict.index(tokens[i]), train_dict.index(tokens[i+2])])

train_set = np.random.permutation(np.array(train_set))

The next step is to set the training configuration. We set the the negative samples size to 10 and the vector size to 100. Learning rate and rate decay are set to 0.1 and 0.995. The training loops are set to 8000000. Logs are displayed each 10000 epoches.

In [24]:
Config = namedtuple("Config", ["dict_size", "vect_size", "neg_samples", "updates", "learning_rate",
                               "learning_rate_decay", "decay_period", "log_period"])
conf = Config(
    dict_size=len(train_dict),
    vect_size=100,
    neg_samples=10,
    updates=8000000,
    learning_rate=0.1,
    learning_rate_decay=0.995,
    decay_period=10000,
    log_period=10000)

We loop over ``updates`` and get the word and context from the train set. We calculate the negative context and calculate the word, context and negative sample vectors. The negative context is chosen randomly. In the next step we calcualte the cost and corresponding to it gradients.

In [25]:
def neg_sample(conf, train_set, train_tokens):
    Vp = randn(conf.dict_size, conf.vect_size)
    Vo = randn(conf.dict_size, conf.vect_size)

    J = 0.0
    learning_rate = conf.learning_rate
    for i in range(conf.updates):
        idx = i % len(train_set)

        word = train_set[idx, 0]
        context = train_set[idx, 1]

        neg_context = np.random.randint(0, len(train_tokens), conf.neg_samples)
        neg_context = train_tokens[neg_context]

        word_vect = Vp[word, :]  # word vector
        context_vect = Vo[context, :];  # context wector
        negative_vects = Vo[neg_context, :]  # sampled negative vectors

        # Cost and gradient calculation starts here
        score_pos = word_vect @ context_vect.T
        score_neg = word_vect @ negative_vects.T

        J -= np.log(sigmoid(score_pos)) + np.sum(np.log(sigmoid(-score_neg)))
        if (i + 1) % conf.log_period == 0:
            print('Update {0}\tcost: {1:>2.2f}'.format(i + 1, J / conf.log_period))
            final_cost = J / conf.log_period
            J = 0.0

        pos_g = 1.0 - sigmoid(score_pos)
        neg_g = sigmoid(score_neg)

        word_grad = -pos_g * context_vect + np.sum(as_matrix(neg_g) * negative_vects, axis=0)
        context_grad = -pos_g * word_vect
        neg_context_grad = as_matrix(neg_g) * as_matrix(word_vect).T

        Vp[word, :] -= learning_rate * word_grad
        Vo[context, :] -= learning_rate * context_grad
        Vo[neg_context, :] -= learning_rate * neg_context_grad

        if i % conf.decay_period == 0:
            learning_rate = learning_rate * conf.learning_rate_decay

    return Vp, Vo, final_cost

Next do the training:

In [26]:
Vp, Vo, J = neg_sample(conf, train_set, train_tokens)

Update 10000	cost: 18.45
Update 20000	cost: 10.67
Update 30000	cost: 8.58
Update 40000	cost: 7.33
Update 50000	cost: 6.67
Update 60000	cost: 6.17
Update 70000	cost: 5.59
Update 80000	cost: 4.80
Update 90000	cost: 4.60
Update 100000	cost: 4.47
Update 110000	cost: 4.37
Update 120000	cost: 4.24
Update 130000	cost: 4.15
Update 140000	cost: 4.00
Update 150000	cost: 3.79
Update 160000	cost: 3.74
Update 170000	cost: 3.71
Update 180000	cost: 3.67
Update 190000	cost: 3.59
Update 200000	cost: 3.55
Update 210000	cost: 3.46
Update 220000	cost: 3.37
Update 230000	cost: 3.39
Update 240000	cost: 3.35
Update 250000	cost: 3.35
Update 260000	cost: 3.27
Update 270000	cost: 3.27
Update 280000	cost: 3.19
Update 290000	cost: 3.16
Update 300000	cost: 3.15
Update 310000	cost: 3.13
Update 320000	cost: 3.12
Update 330000	cost: 3.08
Update 340000	cost: 3.03
Update 350000	cost: 3.02
Update 360000	cost: 2.98
Update 370000	cost: 2.98
Update 380000	cost: 2.97
Update 390000	cost: 2.93
Update 400000	cost: 2.92
Update 

The ``similar_words`` can be used to find related words of the ``word``.

In [27]:
def lookup_word_idx(word, word_dict):
    try:
        return np.argwhere(np.array(word_dict) == word)[0][0]
    except:
        raise Exception("No such word in dict: {}".format(word))

def similar_words(embeddings, word, word_dict, hits):
    word_idx = lookup_word_idx(word, word_dict)
    similarity_scores = embeddings @ embeddings[word_idx]
    similar_word_idxs = np.argsort(-similarity_scores)    
    return [word_dict[i] for i in similar_word_idxs[:hits]]

In [28]:
print('\n\nTraining cost: {0:>2.2f}\n\n'.format(J))

sample_words = ['knight', 'holy', 'grail']

Vp_norm = Vp / as_matrix(np.linalg.norm(Vp , axis=1))
for w in sample_words:
    similar = similar_words(Vp_norm, w, train_dict, 5)
    print('Words similar to {}: {}'.format(w, ", ".join(similar)))



Training cost: 1.91


Words similar to knight: knight, mumble, pound, general, have
Words similar to holy: holy, twong, siren, through, throat
Words similar to grail: grail, doing, one, unladen, mayhem


/var/folders/ly/t71lcvj913l2v610608ynmdr0000gn/T/ipykernel_37986/3093653581.py:9: RuntimeWarning: divide by zero encountered in matmul
  similarity_scores = embeddings @ embeddings[word_idx]
/var/folders/ly/t71lcvj913l2v610608ynmdr0000gn/T/ipykernel_37986/3093653581.py:9: RuntimeWarning: overflow encountered in matmul
  similarity_scores = embeddings @ embeddings[word_idx]
/var/folders/ly/t71lcvj913l2v610608ynmdr0000gn/T/ipykernel_37986/3093653581.py:9: RuntimeWarning: invalid value encountered in matmul
  similarity_scores = embeddings @ embeddings[word_idx]


#### References

[1] Jeffrey Pennington, Richard Socher, and Christopher D. Manning. 2014. GloVe: Global Vectors for Word Representation. 